In [0]:
# Load the raw data
df_raw = spark.read.table("default.sales_data_raw")

# Check what we have
print(f"Total rows: {df_raw.count()}")
df_raw.printSchema()
display(df_raw.limit(5))

Total rows: 10000
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- status: string (nullable = true)



order_id,order_date,customer_id,sales_rep,product,category,region,quantity,unit_price,discount,status
ORD_000001,2023-03-31,CUST_0053,REP_038,Monitor,Electronics,Central,5,161.25,0.0,Completed
ORD_000002,2024-03-05,CUST_0014,REP_015,Desk Chair,Electronics,Central,2,219.07,0.0,Completed
ORD_000003,2023-12-11,CUST_0175,REP_007,USB Hub,Electronics,South,1,53.3,0.2,Completed
ORD_000004,2024-07-03,CUST_0414,REP_008,Monitor,Electronics,West,6,320.97,0.2,Completed
ORD_000005,2023-02-16,CUST_0186,REP_043,Desk Chair,Electronics,North,5,247.28,0.2,Pending


In [0]:
from pyspark.sql import functions as F

# 1. Drop rows where critical fields are null
df_clean = df_raw.dropna(subset=["customer_id", "region"])

# 2. Add derived columns
df_clean = df_clean.withColumn(
    "gross_revenue", F.round(F.col("quantity") * F.col("unit_price"), 2)
)
df_clean = df_clean.withColumn(
    "net_revenue", F.round(F.col("gross_revenue") * (1 - F.col("discount")), 2)
)
df_clean = df_clean.withColumn("order_year",  F.year("order_date"))
df_clean = df_clean.withColumn("order_month", F.month("order_date"))

# 3. Filter out Pending/Returned orders — keep only Completed
df_clean = df_clean.filter(F.col("status") == "Completed")

# Sanity check
print(f"Rows after cleaning: {df_clean.count()}")
display(df_clean.limit(5))

Rows after cleaning: 5675


order_id,order_date,customer_id,sales_rep,product,category,region,quantity,unit_price,discount,status,gross_revenue,net_revenue,order_year,order_month
ORD_000001,2023-03-31,CUST_0053,REP_038,Monitor,Electronics,Central,5,161.25,0.0,Completed,806.25,806.25,2023,3
ORD_000002,2024-03-05,CUST_0014,REP_015,Desk Chair,Electronics,Central,2,219.07,0.0,Completed,438.14,438.14,2024,3
ORD_000003,2023-12-11,CUST_0175,REP_007,USB Hub,Electronics,South,1,53.3,0.2,Completed,53.3,42.64,2023,12
ORD_000004,2024-07-03,CUST_0414,REP_008,Monitor,Electronics,West,6,320.97,0.2,Completed,1925.82,1540.66,2024,7
ORD_000006,2023-06-16,CUST_0195,REP_024,Mouse,Electronics,East,2,81.85,0.2,Completed,163.7,130.96,2023,6


In [0]:
# Save cleaned data as a Delta table
df_clean.write.format("delta").mode("overwrite").saveAsTable("default.sales_data_clean")

print("✅ Clean table saved!")


✅ Clean table saved!


In [0]:
display(df_clean)

order_id,order_date,customer_id,sales_rep,product,category,region,quantity,unit_price,discount,status,gross_revenue,net_revenue,order_year,order_month
ORD_000001,2023-03-31,CUST_0053,REP_038,Monitor,Electronics,Central,5,161.25,0.0,Completed,806.25,806.25,2023,3
ORD_000002,2024-03-05,CUST_0014,REP_015,Desk Chair,Electronics,Central,2,219.07,0.0,Completed,438.14,438.14,2024,3
ORD_000003,2023-12-11,CUST_0175,REP_007,USB Hub,Electronics,South,1,53.3,0.2,Completed,53.3,42.64,2023,12
ORD_000004,2024-07-03,CUST_0414,REP_008,Monitor,Electronics,West,6,320.97,0.2,Completed,1925.82,1540.66,2024,7
ORD_000006,2023-06-16,CUST_0195,REP_024,Mouse,Electronics,East,2,81.85,0.2,Completed,163.7,130.96,2023,6
ORD_000007,2023-06-17,CUST_0088,REP_030,Webcam,Electronics,South,5,73.52,0.15,Completed,367.6,312.46,2023,6
ORD_000008,2023-11-20,CUST_0398,REP_026,Desk Chair,Electronics,North,9,361.97,0.0,Completed,3257.73,3257.73,2023,11
ORD_000009,2023-05-27,CUST_0256,REP_017,Headphones,Electronics,West,10,66.55,0.15,Completed,665.5,565.68,2023,5
ORD_000010,2024-01-06,CUST_0220,REP_015,Keyboard,Electronics,West,9,59.6,0.1,Completed,536.4,482.76,2024,1
ORD_000011,2024-09-02,CUST_0079,REP_005,Keyboard,Electronics,West,2,91.14,0.2,Completed,182.28,145.82,2024,9
